Tuple = (Actor, Action, Object, Condition)

Metrics:

Exact Match F1

Span Match F1 (token overlap)

Semantic Match F1 (SentenceTransformer)

And simple error analysis:

Missing tuples (LLM missed)

Extra tuples (LLM hallucinated)

No component-level metrics.

**Install dependencies**

In [ ]:
!pip install pandas openpyxl sentence-transformers

**Imports and semantic model**

In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# semantic similarity model
model = SentenceTransformer("all-MiniLM-L6-v2")

Parse annotation into tuples

Each argument unit becomes:

(actor, action, object, condition) **bold text**

In [ ]:
def parse_annotation(text):
    """
    Convert annotation text into list of tuples:
    (actor, action, object, condition)
    """

    tuples = []

    if not isinstance(text, str):
        return tuples

    current = {"actor":"", "action":"", "object":"", "condition":""}

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        # remove numbering like "1."
        line = re.sub(r"^\d+\.\s*", "", line)

        m = re.match(r"(Actor|Action|Object|Condition):\s*(.*)", line, re.I)

        if m:

            comp = m.group(1).lower()
            span = m.group(2).strip()

            # new tuple starts when a new Actor appears
            if comp == "actor":
                if any(current.values()):
                    tuples.append(current)
                    current = {"actor":"", "action":"", "object":"", "condition":""}

            current[comp] = span

    if any(current.values()):
        tuples.append(current)

    return tuples

**Matching functions**

In [ ]:
def exact_match(a,b):
    return a.strip().lower() == b.strip().lower()

In [ ]:
import re
import nltk
from collections import Counter
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

lemmatizer = WordNetLemmatizer()

def token_overlap_old(a: str, b: str) -> float:
    """
    Compute normalized token overlap (token-level F1) between two text spans.
    Counts duplicate tokens. Returns a value between 0.0 and 1.0.

    Text is normalized by:
        - Lowercasing
        - Removing punctuation
        - Stripping extra whitespace
        - Lemmatization
    """
    # Normalize text
    def normalize(text):
        text = text.lower()
        text = re.sub(r'[\W_]+', '', text) # More robust punctuation removal
        text = text.strip()
        return text

    def tokenize(text):
        tokens = nltk.word_tokenize(text)
        # Lemmatize each token for robustness
        tokens = [lemmatizer.lemmatize(tok) for tok in tokens]
        return Counter(tokens)

    a_tokens = tokenize(normalize(a))
    b_tokens = tokenize(normalize(b))

    if not a_tokens or not b_tokens:
        return 0.0

    overlap_count = sum((a_tokens & b_tokens).values())
    normalized_overlap = (2 * overlap_count) / (sum(a_tokens.values()) + sum(b_tokens.values()))
    return normalized_overlap

In [ ]:
import re

def token_overlap(a: str, b: str, threshold=0.5):
    """
    Relaxed span match based on token overlap.
    Returns True if overlap proportion exceeds threshold.

    - Converts text to lowercase
    - Removes punctuation
    - Computes overlap as |tokens_a ∩ tokens_b| / max(|tokens_a|, |tokens_b|)
    """
    # Normalize
    def norm(text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = text.strip()
        return text

    a_norm = norm(a)
    b_norm = norm(b)

    tokens_a = set(a_norm.split())
    tokens_b = set(b_norm.split())

    if not tokens_a or not tokens_b:
        return False

    overlap = tokens_a & tokens_b
    score = len(overlap) / max(len(tokens_a), len(tokens_b))

    return score >= threshold

In [ ]:
def tuple_match_threshold(g, p, match_func, threshold=0.7):
    """
    Boolean tuple match using weighted Action + Object scoring.

    - Action (main verb) is mandatory.
    - Object is optional but contributes to score.
    - Returns True if weighted score >= threshold, False otherwise.

    Args:
        g (dict): gold tuple
        p (dict): predicted tuple
        match_func (callable): Exact / TokenOverlap / Semantic match function
        threshold (float): weighted score threshold for TP

    Returns:
        bool: True if tuple considered a match
    """
    # --- Extract / preprocess fields ---
    g_action =  extract_main_verb(g.get("action", ""))
    p_action = extract_main_verb(p.get("action", ""))
    #p_action = p.get("action", "")

    # Action must match
    if not match_func(g_action, p_action):
        return False

    # Object preprocessing
    g_obj = preprocess_text(g.get("object", ""))
    p_obj = preprocess_text(p.get("object", ""))

    action_score = 1.0  # already matched
    object_score = 1.0 if match_func(g_obj, p_obj) else 0.0

    weighted_score = 0.7 * action_score + 0.3 * object_score

    return weighted_score >= threshold

In [ ]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

STOPWORDS = {
    "a","an","the","for","of","to","in","on","at",
    "with","by","and","or","is","are","be"
}

def semantic_match(a, b, threshold=0.7):

    def normalize(text):

        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)

        tokens = [
            w for w in text.split()
            if w not in STOPWORDS
        ]

        return " ".join(tokens)

    a_norm = normalize(a)
    b_norm = normalize(b)

    emb = model.encode([a_norm, b_norm])

    emb = np.array([v/np.linalg.norm(v) for v in emb])

    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]

    return sim >= threshold

Tuple matching

A tuple matches if all components match.

In [ ]:
def span_match(a, b):
    return a.strip().lower() == b.strip().lower()

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Actor normalization mapping
ACTOR_MAP = {
    "you": "facility",
    "the producer": "facility",
    "the owner": "facility",
    "facility": "facility"
}

def normalize_actor(text):
    """Normalize actor aliases to a canonical form."""
    t = text.lower().strip()
    return ACTOR_MAP.get(t, t)



def extract_main_verb(action_text):
    """
    Extract the main verb from a sentence using spacy.
    Returns lemmatized root verb, or original text if no verb found.
    """
    doc = nlp(action_text)
    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ == "VERB":
            return token.lemma_
    return action_text

def preprocess_text(text: str) -> str:
    """
    Normalize text for component-level matching:
    - Lowercase
    - Remove punctuation
    - Strip extra whitespace
    """
    if not text:
        return ""

    text = text.lower()                     # lowercase
    text = re.sub(r'[^\w\s]', '', text)    # remove punctuation
    text = text.strip()                     # remove leading/trailing whitespace
    return text

def tuple_match_priority_llm(g, p, match_func, must_match=["action"], actor_optional=True):
    """
    Match tuples with:
      - Action (main verb) as must-match
      - Actor optional (normalized)
      - Object/Condition optional (preprocessed)

    Notes:
    - Action: extract main verb before comparison
    - Object/Condition: preprocessed for relaxed matching
    - Actor: normalized, optional
    """

    # --- Must-match components ---
    for c in must_match:
        g_val = g.get(c, "")
        p_val = p.get(c, "")

        if c == "action":
            g_val = extract_main_verb(g_val)
            p_val = extract_main_verb(p_val)

        if not match_func(g_val, p_val):
            return False  # must-match failed → tuple fails

    # --- Optional Actor ---
    if actor_optional:
        g_val = normalize_actor(g.get("actor", ""))
        p_val = normalize_actor(p.get("actor", ""))
        _ = match_func(g_val, p_val)  # optional, does not block tuple

    # --- Optional Object and Condition ---
    for c in ["object", "condition"]:
        g_val = preprocess_text(g.get(c, ""))
        p_val = preprocess_text(p.get(c, ""))
        _ = match_func(g_val, p_val)  # optional, does not block tuple

    return True

**Tuple-level evaluation**

Definitions:

TP: predicted tuple matches a gold tuple

FP: predicted tuple with no matching gold tuple

FN: gold tuple not predicted

In [ ]:
def tuple_eval(gold_data, pred_data, match_func):
    """
    Tuple-level evaluation using LLM-aware tuple match:
      - Action: main verb must-match
      - Actor: normalized, optional
      - Object/Condition: optional
    Computes precision, recall, F1.
    """
    tp, fp, fn = 0, 0, 0

    for ex in gold_data:
        gold = gold_data[ex]
        pred = pred_data.get(ex, [])
        matched = set()

        for p_tuple in pred:
            found = False
            for i, g_tuple in enumerate(gold):
                if i in matched:
                    continue
                # Use LLM-aware tuple match here
                if tuple_match_threshold(g_tuple, p_tuple, match_func):
                    tp += 1
                    matched.add(i)
                    found = True
                    break
            if not found:
                fp += 1

        # Remaining gold tuples are false negatives
        fn += len(gold) - len(matched)

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return precision, recall, f1

**Simple error analysis**
Tracks:

Missing tuples

Extra tuples

In [ ]:
def error_analysis_extended(gold_data, pred_data, match_func, threshold=0.7):
    """
    Returns:
        missing: list of (example_id, gold_tuple, missing_components)
        extra: list of (example_id, pred_tuple, extra_components)
    missing_components / extra_components: list of fields that did not match
    """
    missing = []
    extra = []

    for ex in gold_data:
        gold = gold_data[ex]
        pred = pred_data.get(ex, [])
        matched = set()

        # --- Check missing gold tuples ---
        for g in gold:
            found = False
            for j, p in enumerate(pred):
                if tuple_match_threshold(g, p, match_func, threshold=threshold):
                    matched.add(j)
                    found = True
                    break
            if not found:
                # Identify which components are failing
                comp_miss = []
                for c in ["action", "object", "actor", "condition"]:
                    g_val = g.get(c, "")
                    p_vals = [p.get(c, "") for p in pred]
                    if not any(match_func(g_val, pv) for pv in p_vals):
                        comp_miss.append(c)
                missing.append((ex, g, comp_miss))

        # --- Check extra predicted tuples ---
        for j, p in enumerate(pred):
            if j not in matched:
                # Identify which components are extra (not matching any gold)
                comp_extra = []
                for c in ["action", "object", "actor", "condition"]:
                    p_val = p.get(c, "")
                    g_vals = [g.get(c, "") for g in gold]
                    if not any(match_func(p_val, gv) for gv in g_vals):
                        comp_extra.append(c)
                extra.append((ex, p, comp_extra))

    return missing, extra

Component Level

In [ ]:
# =========================
# Component-level evaluation for AAOC
# =========================

def component_eval(gold_data, pred_data, match_func):
    """
    Computes precision, recall, F1 per tuple component:
      - Action: main verb extraction
      - Actor: normalized, optional
      - Object/Condition: optional
    """
    metrics = {c: [0,0,0] for c in ["actor","action","object","condition"]}

    for ex in gold_data:
        gold = gold_data[ex]
        pred = pred_data.get(ex, [])

        for comp in metrics:
            gold_spans = [g[comp] for g in gold]
            pred_spans = [p[comp] for p in pred]

            # Normalize / extract main verb
            # Normalize / extract main verb
            '''
            if comp == "actor":
                gold_spans = [normalize_actor(x) for x in gold_spans]
                pred_spans = [normalize_actor(x) for x in pred_spans]
            elif comp == "action":
                gold_spans = [extract_main_verb(x) for x in gold_spans]
                pred_spans = [extract_main_verb(x) for x in pred_spans]
            else:  # Object / Condition
                gold_spans = [preprocess_text(x) for x in gold_spans]
                pred_spans = [preprocess_text(x) for x in pred_spans]

            '''

            matched = set()
            # Count TP/FP
            for p_span in pred_spans:
                found = False
                for i, g_span in enumerate(gold_spans):
                    if i in matched:
                        continue
                    if match_func(g_span, p_span):
                        metrics[comp][0] += 1  # TP
                        matched.add(i)
                        found = True
                        break
                if not found:
                    metrics[comp][1] += 1  # FP
            # Count FN
            metrics[comp][2] += len(gold_spans) - len(matched)

    # Compute precision, recall, F1
    results = {}
    for c, (tp, fp, fn) in metrics.items():
        precision = tp/(tp+fp) if tp+fp>0 else 0
        recall = tp/(tp+fn) if tp+fn>0 else 0
        f1 = 2*precision*recall/(precision+recall) if precision+recall>0 else 0
        results[c] = (precision, recall, f1)

    return results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from openpyxl import load_workbook

# ========================================
# Excel configuration
# ========================================
GOLD_FILE = '/content/drive/MyDrive/ArgMining 2026/Legal Text Extraction_All.xlsx'
LLM_FILE = GOLD_FILE

SHEET_NAME = "New Text"  # or specify sheet name

GOLD_ID_COLUMN = "B"
GOLD_COLUMN = "E"

LLM_ID_COLUMN = "B"
#LLM_PRED_COLUMN = "F"# gpt-oss
#LLM_PRED_COLUMN = "G"# phi
LLM_PRED_COLUMN = "H"# AdaptLaw


# ========================================
# Excel processing helpers
# ========================================
def detect_start_row(worksheet, input_column: str) -> int:
    first_value = worksheet[f"{input_column}1"].value
    if isinstance(first_value, str) and first_value.strip().lower() in {
        "excerpt", "excerpts", "text", "input", "excerpt_id"
    }:
        return 2
    return 1


# ========================================
# Main processing
# ========================================
def process_excel_files():

    gold_data = {}
    pred_data = {}

    # ---- Load GOLD file ----
    gold_workbook = load_workbook(GOLD_FILE)
    gold_ws = gold_workbook[SHEET_NAME] if SHEET_NAME else gold_workbook.active
    gold_start_row = detect_start_row(gold_ws, GOLD_ID_COLUMN)

    # ---- Load LLM file ----
    llm_workbook = load_workbook(LLM_FILE)
    llm_ws = llm_workbook[SHEET_NAME] if SHEET_NAME else llm_workbook.active
    llm_start_row = detect_start_row(llm_ws, LLM_ID_COLUMN)

    # ========================================
    # Process GOLD rows
    # ========================================
    for row_idx in range(gold_start_row, gold_ws.max_row + 1):

        excerpt_id = gold_ws[f"{GOLD_ID_COLUMN}{row_idx}"].value
        gold_annotation = gold_ws[f"{GOLD_COLUMN}{row_idx}"].value

        ## By EF


        excerpt_id_str = str(excerpt_id).strip() if excerpt_id else ""
        gold_str = str(gold_annotation).strip() if gold_annotation else ""

        ##by EF


        if not excerpt_id_str or not gold_str:
            continue

        try:
            gold_data[excerpt_id_str] = parse_annotation(gold_str)
        except Exception as e:
            print(f"GOLD parse error row {row_idx}: {e}")

        ## by EF
        #print("Gold annotate parsed ", gold_data[excerpt_id_str])
        #break

    # ========================================
    # Process LLM rows
    # ========================================
    for row_idx in range(llm_start_row, llm_ws.max_row + 1):

        excerpt_id = llm_ws[f"{LLM_ID_COLUMN}{row_idx}"].value
        prediction = llm_ws[f"{LLM_PRED_COLUMN}{row_idx}"].value

        ##by EF


        excerpt_id_str = str(excerpt_id).strip() if excerpt_id else ""
        pred_str = str(prediction).strip() if prediction else ""

        if not excerpt_id_str or not pred_str:
            continue

        try:
            pred_data[excerpt_id_str] = parse_annotation(pred_str)
        except Exception as e:
            print(f"LLM parse error row {row_idx}: {e}")

        ## by EF
        #print("pred annotate parsed ", pred_data[excerpt_id_str])
        #break

    print(f"Loaded {len(gold_data)} gold annotations")
    print(f"Loaded {len(pred_data)} LLM predictions")

    return gold_data, pred_data

In [ ]:
### gold data, pred_data by llm
gold_data, pred_data = process_excel_files()

In [ ]:
def tuple_score(g, p, match_func):
    """
    Compute a simple similarity score between a gold tuple g and predicted tuple p.
    Uses Action + Object as main scoring fields.
    Returns a score in [0,1].
    """
    # Action: main verb extraction
    g_action = extract_main_verb(g.get("action", ""))
    p_action = extract_main_verb(p.get("action", ""))

    action_score = 1.0 if match_func(g_action, p_action) else 0.0

    # Object: preprocessed, token/semantic match
    g_obj = preprocess_text(g.get("object", ""))
    p_obj = preprocess_text(p.get("object", ""))
    object_score = 1.0 if match_func(g_obj, p_obj) else 0.0

    # Weighted sum (can adjust weights)
    return 0.7 * action_score + 0.3 * object_score


def filter_llm_output(pred_tuples, gold_tuples, match_func):
    """
    Select predicted tuples that best match gold tuples.
    Returns up to len(gold_tuples) predicted tuples.
    """
    scores = []
    for p in pred_tuples:
        # score against all gold tuples, take max
        s = max([tuple_score(g, p, match_func) for g in gold_tuples]) if gold_tuples else 0
        scores.append(s)

    # Sort predicted tuples by score descending
    sorted_preds = [p for _, p in sorted(zip(scores, pred_tuples), key=lambda x: x[0], reverse=True)]

    # Take top N tuples
    n = min(len(gold_tuples)*2, len(pred_tuples))
    return sorted_preds[:n]


In [ ]:
FACILITY_TERMS = [
    "person", "persons", "owner", "operator", "manufacturer",
    "entity", "establishment", "facility", "producer", "you"
]

def normalize_actor_pred(actor_text: str) -> str:
    """
    Normalize LLM-predicted actors:
    Map any mention of person/owner/manufacturer/entity to 'Facility'.
    """
    if not actor_text:
        return ""

    text = actor_text.lower().strip()

    if any(term in text for term in FACILITY_TERMS):
        return "Facility"

    return actor_text.strip()

Run Eval

In [ ]:
methods = {
    #"Exact": exact_match,
    "TokenOverlap": token_overlap,
    "Semantic": semantic_match
}

for name, func in methods.items():

    print("\n==============================")
    print(name, "MATCH")
    print("==============================")

    # --- Filter predicted tuples for this match function ---
    # --- Filter and preprocess predicted tuples ---
    pred_data_processed = {}
    for ex in gold_data:

        # Get raw predicted tuples
        raw_pred = pred_data.get(ex, [])

        # Preprocess: normalize actor, extract main verb, preprocess object/condition
        processed_pred = []
        for t in raw_pred:
            new_t = {}
            # Action: extract main verb
            new_t['action'] = extract_main_verb(t.get('action', ''))

            # Actor: normalize
            new_t['actor'] = normalize_actor_pred(t.get('actor', ''))

            # Object / Condition: preprocess (lower, strip, remove punctuation)
            new_t['object'] = preprocess_text(t.get('object', ''))
            new_t['condition'] = preprocess_text(t.get('condition', ''))

            processed_pred.append(new_t)

        # Optional: shorten predicted list to match gold tuples
        pred_data_processed[ex] = filter_llm_output(
            processed_pred,
            gold_data[ex],
            func
        )

    p,r,f = tuple_eval(gold_data, pred_data_processed, func)

    print("Precision:", round(p,3))
    print("Recall:", round(r,3))
    print("Tuple F1:", round(f,3))

    #missing, extra = error_analysis_extended(gold_data, pred_data_processed, func)

    #print("\nError Analysis")
    #print("Missing tuples examples:", missing[:1])
    #print("Extra tuples examples:", extra[:1])

    comp_results = component_eval(gold_data, pred_data_processed, func)
    print("gold ", gold_data)
    print("pred ", pred_data)
    print("\nComponent-level F1:", name)
    for c in ["actor","action","object","condition"]:
      print(f"{c.capitalize():<9} F1: {round(comp_results[c][2],3)}")